# C-B2 生物学特征校正：滚动测试

本 notebook 调用 `src/fitness/c_b2_biological_correction_demo.py`，先修正 C-B2 的未来分支计数，再将计数归一化为频率。Fitness 只在分支原始平均抗原距离大于 2 时启用，作为抗原逃逸后的条件性修正。

运行测试会更新 `outputs/h1n1_c_b2_biological_correction_past_strain_3y_fitness_gated_at_2/` 中的 CSV 和 JSON 结果文件。计数预测使用 `C-B2_count × exp(t × relative_fitness)`，随后按每个 14 天 bin 归一化为频率；现在九个截止点均有抗原性评分。在前三个截止点用于训练后，滚动评估覆盖 2023-03-31 至 2025-09-30；除 `mean_top1_accuracy` 外，所有指标均是**越低越好**。

In [ ]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path('/home/chenyh/workspace/Alright-strain')
SCRIPT = ROOT / 'src' / 'fitness' / 'c_b2_biological_correction_demo.py'
SCORES_SQLITE = ROOT / 'outputs' / 'h1n1_antigenicity' / 'past_strain_3y_full_excluding_recent6m.sqlite'
CUTOFFS_CSV = ROOT / 'outputs' / 'h1n1_antigenicity' / 'panels_2017_2025_past_strain_3y' / 'forecast_cutoffs.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'h1n1_c_b2_biological_correction_past_strain_3y_full_count_to_frequency_gated_at_2'

assert SCRIPT.is_file(), f'找不到测试脚本：{SCRIPT}'
assert SCORES_SQLITE.is_file(), f'找不到新的抗原性数据：{SCORES_SQLITE}'
assert CUTOFFS_CSV.is_file(), f'找不到截止点配置：{CUTOFFS_CSV}'

cutoffs = pd.read_csv(CUTOFFS_CSV)['forecast_cutoff'].astype(str).tolist()
with sqlite3.connect(SCORES_SQLITE) as connection:
    scored_cutoffs = set(pd.read_sql_query(
        'SELECT DISTINCT forecast_cutoff FROM query_summary', connection
    )['forecast_cutoff'].astype(str))
missing_cutoffs = sorted(set(cutoffs) - scored_cutoffs)
assert not missing_cutoffs, f'抗原性评分尚未完成的截止点：{missing_cutoffs}'
print(f'评分覆盖完整：{len(cutoffs)} 个截止点（{cutoffs[0]} 至 {cutoffs[-1]}）。')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)


## 1. 配置并运行测试

本版使用过去 3 年（剔除最近 6 个月）的病毒-病毒抗原距离数据：`past_strain_3y_full_excluding_recent6m.sqlite`。`--overwrite` 只会覆盖本次独立输出目录，不影响血清面板版本的结果。

In [ ]:
MIN_SCORE_SEQUENCES = 10
MIN_TRAIN_CUTOFFS = 3
L2 = 1.0
FITNESS_ANTIGENICITY_THRESHOLD = 2.0
PREDICTION_TARGET = 'count_to_frequency'  # 先预测 count，再归一化为频率
OBJECTIVE = 'top1'  # 'top1' 或 'distribution'，在频率空间训练与评估
TRAJECTORY_MODE = 'relative_fitness'  # 'relative_fitness' 或 'static'

command = [
    sys.executable, str(SCRIPT),
    '--scores-sqlite', str(SCORES_SQLITE),
    '--cutoffs-csv', str(CUTOFFS_CSV),
    '--min-score-sequences', str(MIN_SCORE_SEQUENCES),
    '--min-train-cutoffs', str(MIN_TRAIN_CUTOFFS),
    '--l2', str(L2),
    '--fitness-antigenicity-threshold', str(FITNESS_ANTIGENICITY_THRESHOLD),
    '--objective', OBJECTIVE,
    '--prediction-target', PREDICTION_TARGET,
    '--trajectory-mode', TRAJECTORY_MODE,
    '--output-dir', str(OUTPUT_DIR),
    '--overwrite',
]

run = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
if run.returncode != 0:
    raise RuntimeError(f'测试运行失败：\n{run.stderr}')
report = json.loads(run.stdout)
baseline_cutoffs = pd.read_csv(OUTPUT_DIR / 'c_b2_all_nine_windows.csv')['forecast_cutoff'].astype(str).drop_duplicates().sort_values().tolist()
expected_tested_cutoffs = baseline_cutoffs[MIN_TRAIN_CUTOFFS:]
assert report['tested_cutoffs'] == expected_tested_cutoffs, (
    f"滚动测试未覆盖完整可测截止点：{report['tested_cutoffs']}；期望：{expected_tested_cutoffs}"
)
print('测试完成。输出目录：', OUTPUT_DIR)
print('可评估截止点：', ', '.join(report['tested_cutoffs']))


## 2. 汇总结果

滚动评估中的每一个截止点只使用更早的截止点训练校正系数，因此不会泄漏未来数据。

In [ ]:
summary = pd.read_csv(OUTPUT_DIR / 'rolling_summary.csv')
detail = pd.read_csv(OUTPUT_DIR / 'rolling_metrics.csv')
coefficients = pd.read_csv(OUTPUT_DIR / 'rolling_coefficients.csv')
baseline_all = pd.read_csv(OUTPUT_DIR / 'c_b2_all_nine_windows.csv')

display(summary.style.format({
    'mean_MAE': '{:.4f}', 'mean_RMSE': '{:.4f}', 'mean_TVD': '{:.4f}',
    'mean_JSD': '{:.4f}', 'mean_top1_accuracy': '{:.1%}',
}))

best_tvd = summary.loc[summary['mean_TVD'].idxmin()]
print(f"按平均 TVD 最佳：{best_tvd['model']} ({best_tvd['mean_TVD']:.4f})")
print(f"完整 9 个窗口的未校正 C-B2 平均 TVD：{baseline_all['mean_TVD'].mean():.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
plot_order = summary.sort_values('mean_TVD')['model']
sns.barplot(data=summary, y='model', x='mean_TVD', order=plot_order, ax=axes[0], color='#4C78A8')
axes[0].set(title='滚动测试：平均 TVD（越低越好）', xlabel='mean_TVD', ylabel='')
sns.barplot(data=summary, y='model', x='mean_top1_accuracy', order=plot_order, ax=axes[1], color='#F58518')
axes[1].set(title='滚动测试：Top-1 准确率（越高越好）', xlabel='mean_top1_accuracy', ylabel='')
axes[1].xaxis.set_major_formatter('{x:.0%}')
plt.show()


## 3. 按截止点的明细与拟合系数

用下表定位特征在哪些截止点改善或恶化预测；系数是在每个测试截止点之前的历史窗口上拟合的。

In [ ]:
detail_view = detail.sort_values(['forecast_cutoff', 'mean_TVD']).copy()
display(detail_view.style.format({
    'MAE': '{:.4f}', 'RMSE': '{:.4f}', 'mean_TVD': '{:.4f}',
    'mean_JSD': '{:.4f}', 'top1_accuracy': '{:.1%}',
}))

if coefficients.empty:
    print('基线 C-B2 没有待拟合的校正系数。')
else:
    coefficient_table = coefficients.pivot(
        index='forecast_cutoff', columns=['model', 'feature'], values='coefficient'
    ).sort_index()
    display(coefficient_table.style.format('{:.4f}'))
